# Week 21, Day 6: Forge: Week 21 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w021/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("1. tau = RC =", 100e6 * 200e-12 * 1000, "ms")
print("2. dt below 2 tau =", 2 * 20, "ms")
print("3.", round(1000 / (2 + 20 * np.log(30 / (30 - 15))), 1), "Hz")

In [ ]:
gNa, gK, gL, ENa, EK, EL, C = 120.0, 36.0, 0.3, 50.0, -77.0, -54.387, 1.0

def hh_rates(V):
    an = 0.01 * (V + 55) / (1 - np.exp(-(V + 55) / 10)); bn = 0.125 * np.exp(-(V + 65) / 80)
    am = 0.1 * (V + 40) / (1 - np.exp(-(V + 40) / 10)); bm = 4.0 * np.exp(-(V + 65) / 18)
    ah = 0.07 * np.exp(-(V + 65) / 20); bh = 1 / (1 + np.exp(-(V + 35) / 10))
    return an, bn, am, bm, ah, bh

def hh_rest():
    an, bn, am, bm, ah, bh = hh_rates(-65.0)
    return np.array([-65.0, an / (an + bn), am / (am + bm), ah / (ah + bh)])

def count_spikes(V):
    return int(np.sum((V[1:] > 0) & (V[:-1] <= 0)))      # upward crossings of 0 mV
print("resting state (V, n, m, h):", hh_rest().round(3))

### Exercise: Part A: the Hodgkin-Huxley neuron fires

Complete `hh_rhs(y, I)`, which returns the derivatives of (V, n, m, h), and the RK4 step in `simulate(Ifun, T)`, which returns the voltage at every step. Store the voltage for the 10 μA/cm² step in `V_A`.

In [ ]:
def hh_rhs(y, I):
    V, n, m, h = y
    an, bn, am, bm, ah, bh = hh_rates(V)
    dV = ...
    return np.array([dV, an * (1 - n) - bn * n, am * (1 - m) - bm * m, ah * (1 - h) - bh * h])

def simulate(Ifun, T, dt=0.01, y0=None):
    y = hh_rest() if y0 is None else y0
    Vs = np.empty(int(T / dt))
    for i in range(len(Vs)):
        t = i * dt
        k1 = hh_rhs(y, Ifun(t)); k2 = hh_rhs(y + dt / 2 * k1, Ifun(t + dt / 2))
        k3 = ...; k4 = ...
        y = y + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        Vs[i] = y[0]
    return Vs

V_A = simulate(lambda t: 10.0 if 5 <= t < 55 else 0.0, 60.0)
print(f"{count_spikes(V_A)} spikes, peak {V_A.max():.1f} mV, lowest {V_A.min():.1f} mV")
plt.plot(np.arange(len(V_A)) * 0.01, V_A); plt.xlabel("ms"); plt.ylabel("V (mV)"); plt.show()

In [ ]:
import numpy as np
def test_rest():
    "with no input the neuron stays within 0.5 mV of rest for 20 ms"
    V0 = simulate(lambda t: 0.0, 20.0)
    assert np.abs(V0 + 65).max() < 0.5

def test_spikes():
    "with 10 uA/cm2 it fires, peaking above +30 mV, several times in 50 ms"
    assert V_A.max() > 30 and count_spikes(V_A) >= 4

test_rest()
test_spikes()
print("All checks passed")

### Exercise: Part B: the rate jumps

Simulate all the currents in `Is_B` together for 300 ms (y has shape 4 by the number of currents), store the voltage of every neuron at every step, and count upward crossings of 0 mV between 100 and 300 ms to get `rates_B` in Hz.

In [ ]:
Is_B = np.array([0, 2, 4, 6, 6.5, 7, 8, 10, 15, 20.0])
y = np.tile(hh_rest()[:, None], (1, len(Is_B)))
dt, steps = 0.01, 30000
Vs = np.empty((steps, len(Is_B)))
for i in range(steps):
    k1 = hh_rhs(y, Is_B); k2 = hh_rhs(y + dt / 2 * k1, Is_B)
    k3 = hh_rhs(y + dt / 2 * k2, Is_B); k4 = hh_rhs(y + dt * k3, Is_B)
    y = y + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    Vs[i] = y[0]
late = Vs[10000:]
rates_B = ...                                   # crossings per neuron, divided by 0.2 s
print(dict(zip(Is_B.tolist(), rates_B.tolist())), " Hz")
plt.plot(Is_B, rates_B, "o-"); plt.xlabel("I (uA/cm2)"); plt.ylabel("rate (Hz)"); plt.show()

In [ ]:
import numpy as np
def test_silent():
    "the neuron does not fire repetitively at 4 uA/cm2 or below"
    assert all(r == 0 for I, r in zip(Is_B, rates_B) if I <= 4)

def test_jump():
    "the lowest non-zero steady rate is above 40 Hz: there is no slow firing"
    assert rates_B[rates_B > 0].min() > 40

def test_rises():
    "the rate at 20 uA/cm2 is higher than at the onset"
    assert rates_B[-1] > rates_B[rates_B > 0].min()

test_silent()
test_jump()
test_rises()
print("All checks passed")

### Exercise: Part C: refractoriness emerges

Loop over delays 2, 3, ..., 20 ms, simulate 40 ms with both pulses, and store in `d_min` the first delay that gives two spikes.

In [ ]:
d_min = None
for d in range(2, 21):
    pulses = lambda t, d=d: 20.0 if (1 <= t < 2) or (1 + d <= t < 2 + d) else 0.0
    n_spk = ...
    if n_spk == 2:
        d_min = d
        break
print("second spike first appears at a delay of", d_min, "ms")

In [ ]:
import numpy as np
def test_first():
    "a single pulse produces exactly one spike"
    assert count_spikes(simulate(lambda t: 20.0 if 1 <= t < 2 else 0.0, 40.0)) == 1

def test_dmin():
    "the shortest delay that gives a second spike lies between 8 and 14 ms"
    assert 8 <= d_min <= 14

test_first()
test_dmin()
print("All checks passed")